<a href="https://colab.research.google.com/github/anikadivya/data-analytics-assigmnets/blob/main/Week2_Assessment_Anika_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## Indian Startup Funding - Decoded


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

---
# Section A – Interview Theory


### Q1. Series vs DataFrame

A Series is just one column with an index. A DataFrame is a bunch of Series stuck together sharing the same index, basically a table. Something like `.mean()` works on both. Something like selecting two columns `df[['a','b']]` only makes sense on a DataFrame, a Series doesn't have other columns to select.


### Q2. inplace=True

It changes the df directly instead of giving you back a new one, so `df.dropna(inplace=True)` edits df and returns None. Pandas is moving away from it because it doesn't actually save memory the way people think, and it makes chaining/debugging annoying. I'd skip it when I still want the original df around, like `df_clean = df.dropna()` instead.


### Q3. fillna() vs dropna()

fillna replaces missing values, dropna just deletes the rows/cols that have them. Use fillna when you don't want to lose the rest of the row's info, e.g. filling a missing rating with the mean. Use dropna when the missing value makes the row useless anyway, e.g. no startup name at all.


### Q4. pd.to_numeric with errors='coerce'

Without coerce, it just crashes the second it hits something it can't convert, like the word 'Undisclosed'. With coerce it turns that into NaN and keeps going. On messy real data I'd always use coerce, otherwise one bad row kills the whole script.


### Q5. .loc vs .iloc

.loc goes by label, .iloc goes by position. `df.loc['b']` = row labeled b. `df.iloc[1]` = whatever row is physically second. They give different results once the index isn't 0,1,2... anymore — like after sorting, loc[0] still finds the row that was originally labeled 0, iloc[0] just grabs whatever's on top now.


### Q6. .map() vs .apply() vs .replace()

map = simple lookup/transform on a Series (dict or function). apply = same idea but more powerful, can run more complex logic, also works on DataFrames row/col wise. replace = swap specific values for other values. I'd use map for a clean dict mapping (like fixing city names), apply when there's actual conditional logic, replace for quick swaps.


### Q7. SettingWithCopyWarning

Happens when pandas isn't sure if you're editing the real df or some copy of it, usually from chained indexing like `df[df['a']>1]['b'] = 5`. Fix: use .loc directly — `df.loc[df['a']>1, 'b'] = 5` — or copy it first if you actually want a separate df.


### Q8. pd.cut() vs pd.qcut()

cut = you set the bin edges yourself (fixed ranges). qcut = bins based on quantiles so each group has roughly equal count. For Low/Mid/High income I'd use qcut if the data's skewed and I want balanced groups, cut if the business already has set thresholds in mind (like Low = below 5L).

---
# Section B – Predict the Output


### Q9. Prediction:
```
   a  b
0  10  4
1  20  5
2  30  6
```
df2 = df doesn't copy anything, it's the same object with a new name. Editing df2 edits df too.

In [2]:
# Q9
import pandas as pd
df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})
df2 = df
df2['a'] = [10, 20, 30]
print(df)
# right - no copy happened, both point to the same df

    a  b
0  10  4
1  20  5
2  30  6



### Q10. Prediction:
```
3.0
12.0
```
mean/sum both skip NaN by default.

In [3]:
# Q10
import pandas as pd
import numpy as np
s = pd.Series([1, 2, np.nan, 4, 5])
print(s.mean())
print(s.sum())
# correct

3.0
12.0



### Q11. Prediction:
```
float64
700.0
```
'NA' becomes NaN, and float64 because NaN forces the column to be float.

In [4]:
# Q11
import pandas as pd
df = pd.DataFrame({'price': ['100', '200', 'NA', '400']})
df['price'] = pd.to_numeric(df['price'], errors='coerce')
print(df['price'].dtype)
print(df['price'].sum())
# correct

float64
700.0



### Q12. Prediction:
```
name    Rahul
age        22
Name: 0, dtype: object
<class 'pandas.core.series.Series'>
```
One row from .loc comes back as a Series, columns become the index.

In [5]:
# Q12
import pandas as pd
df = pd.DataFrame({'name': ['Rahul', 'Priya'], 'age': [22, 21]})
print(df.loc[0])
print(type(df.loc[0]))
# correct

name    Rahul
age        22
Name: 0, dtype: object
<class 'pandas.core.series.Series'>



### Q13. Prediction:
df stays the same, df_filtered might update or might not, and it throws a SettingWithCopyWarning either way. df_filtered = df[...] could be a view or a copy, pandas doesn't guarantee which, so the next assignment is undefined behavior. Fix: `.copy()` after filtering.

In [6]:
# Q13
import pandas as pd
df = pd.DataFrame({'rating': [4.5, 3.8, 4.2, 5.0]})
df_filtered = df[df['rating'] > 4]
df_filtered['rating'] = df_filtered['rating'] * 2
print(df)
print(df_filtered)
# correct - warning shows up, df untouched here

   rating
0     4.5
1     3.8
2     4.2
3     5.0
   rating
0     9.0
2     8.4
3    10.0


/tmp/ipykernel_3684/204660221.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['rating'] = df_filtered['rating'] * 2



### Q14. Prediction:
```
0      1
1      4
2    small
3    small
Name: a, dtype: object
object
```
dtype is object because the lambda returns a mix of numbers and strings, pandas can't keep that as int/float.

In [7]:
# Q14
import pandas as pd
df = pd.DataFrame({'a': [1, 2, 3, 4]})
result = df['a'].apply(lambda x: x**2 if x > 2 else 'small')
print(result)
print(result.dtype)
# correct

0    small
1    small
2        9
3       16
Name: a, dtype: object
object



### Q15. Prediction:
```
nunique() -> 3 (skips NaN)
len(unique()) -> 4 (unique() keeps NaN as one of the values)
value_counts() -> Delhi 2, Mumbai 1, Bengaluru 1 (skips NaN too)
```

In [8]:
# Q15
import pandas as pd
df = pd.DataFrame({'city': ['Delhi', 'Mumbai', 'Delhi', 'Bengaluru', None]})
print(df['city'].nunique())
print(len(df['city'].unique()))
print(df['city'].value_counts())
# correct

3
4
city
Delhi        2
Mumbai       1
Bengaluru    1
Name: count, dtype: int64



### Q16. Prediction:
First print is fine. The commented line breaks — `&` binds tighter than `>` in Python, so it actually evaluates as `df['a'] > (1 & df['b']) > 15`, not what you want. Need parentheses around each condition, like the third line does.

In [9]:
# Q16
import pandas as pd
df = pd.DataFrame({'a': [1, 2, 3], 'b': [10, 20, 30]})
print(df[df['a'] > 1])
# print(df[df['a'] > 1 & df['b'] > 15])
print(df[(df['a'] > 1) & (df['b'] > 15)])
# correct

   a   b
1  2  20
2  3  30
   a   b
1  2  20
2  3  30


---
# Section C – Applied on Indian Startups Dataset


### Q17. Raw CSV check (before cleaning)

In [10]:
# Q17
df_raw = pd.read_csv('indian_startups_funding.csv')

print("(a) shape:", df_raw.shape)

print("\n(b) missing values:")
print(df_raw.isnull().sum())

print("\n(c) duplicate rows:", df_raw.duplicated().sum())

(a) shape: (468, 8)

(b) missing values:
Startup Name      0
City              0
INDUSTRY          0
Funding Stage     0
Amount            0
Funding Date     19
Investors        10
Founded Year     23
dtype: int64

(c) duplicate rows: 18


### Cleaning — doing it once here, using this df for everything after.

In [11]:
#CLEANING
df = df_raw.copy()

# rename cols
df.columns = ['startup_name', 'city', 'industry', 'funding_stage',
              'amount', 'funding_date', 'investors', 'founded_year']

# clean text cols
df['startup_name'] = df['startup_name'].str.strip().str.title()
df['city'] = df['city'].str.strip().str.title()

# city aliases -> canonical name
city_map = {
    'Ahmedabad': 'Ahmedabad',
    'Bangalore': 'Bengaluru', 'Blr': 'Bengaluru', 'Bengaluru': 'Bengaluru',
    'Bombay': 'Mumbai', 'Bom': 'Mumbai', 'Mumbai': 'Mumbai',
    'Chennai': 'Chennai', 'Madras': 'Chennai',
    'Calcutta': 'Kolkata', 'Kolkata': 'Kolkata',
    'Delhi': 'New Delhi', 'New Delhi': 'New Delhi',
    'Ggn': 'Gurugram', 'Gurgaon': 'Gurugram', 'Gurugram': 'Gurugram',
    'Hyd': 'Hyderabad', 'Hyderabad': 'Hyderabad',
    'Noida': 'Noida',
    'Pune': 'Pune',
}
# note: keys match .title() casing, e.g. "BLR" -> "Blr"
df['city'] = df['city'].map(city_map)

# industry aliases -> canonical name
# normalize first (upper, strip spaces/hyphens/slashes) so all variants collapse to one key
industry_key = (df['industry'].str.strip().str.upper()
                 .str.replace('-', '', regex=False)
                 .str.replace(' ', '', regex=False)
                 .str.replace('/', '', regex=False))
industry_map = {
    'AGRITECH': 'AgriTech',
    'AIML': 'AI/ML',
    'BEAUTYTECH': 'BeautyTech',
    'ECOMMERCE': 'E-commerce',
    'EDTECH': 'EdTech',
    'EV': 'EV', 'ELECTRICVEHICLES': 'EV',
    'FINTECH': 'FinTech', 'FT': 'FinTech',
    'FOODTECH': 'FoodTech',
    'GAMING': 'Gaming', 'GAMES': 'Gaming',
    'HEALTHTECH': 'HealthTech',
    'LOGISTICS': 'Logistics',
    'PROPTECH': 'PropTech',
    'SAAS': 'SaaS',
    'TRAVELTECH': 'TravelTech',
}
df['industry'] = industry_key.map(industry_map)

# funding stage aliases -> canonical name
stage_key = (df['funding_stage'].str.strip().str.upper()
              .str.replace('-', '', regex=False)
              .str.replace('_', '', regex=False)
              .str.replace(' ', '', regex=False))
stage_map = {
    'BRIDGE': 'Bridge',
    'IPO': 'IPO',
    'PREIPO': 'Pre-IPO',
    'PRESEED': 'Pre-Seed',
    'PRESERIESA': 'Pre-Series A', 'PREA': 'Pre-Series A',
    'SEED': 'Seed',
    'SERIESA': 'Series A', 'SERIESB': 'Series B', 'SERIESC': 'Series C',
    'SERIESD': 'Series D', 'SERIESE': 'Series E',
}
df['funding_stage'] = stage_key.map(stage_map)

# amount -> numeric, in Cr
amount_clean = df['amount'].astype(str).str.strip()
amount_clean = amount_clean.replace('Undisclosed', np.nan)
amount_clean = (amount_clean.str.replace('₹', '', regex=False)
                             .str.replace('INR', '', regex=False)
                             .str.replace('Crore', '', regex=False)
                             .str.replace('Cr', '', regex=False)
                             .str.replace(',', '', regex=False)
                             .str.strip())
df['amount'] = pd.to_numeric(amount_clean, errors='coerce')

# date -> datetime
# format='mixed' matters here - the col has 4 different date formats mixed together,
# without it pandas guesses one format from the top rows and quietly breaks the rest
df['funding_date'] = pd.to_datetime(df['funding_date'], errors='coerce',
                                     dayfirst=True, format='mixed')

# drop dupes
df = df.drop_duplicates()

print("cleaned shape:", df.shape)
df.head()

cleaned shape: (450, 8)


,startup_name,city,industry,funding_stage,amount,funding_date,investors,founded_year
0,Playtime Games,New Delhi,E-commerce,Series E,942.0,2024-02-29,"Ranjan Pai, Prosus Ventures",2020.0
1,Rebel Foods,Ahmedabad,BeautyTech,Pre-Series A,1440.0,2018-08-18,Lightspeed,2017.0
2,Freshworks,Kolkata,AgriTech,Pre-Series A,329.0,2020-07-14,Info Edge,2020.0
3,Bigbasket,Mumbai,E-commerce,Series D,1405.0,2018-03-31,B Capital,2016.0
4,Ola,Kolkata,EdTech,Series E,2223.0,2021-08-24,Falcon Edge,2017.0



### Q18. Column names, sorted

In [ ]:
# Q18
print(sorted(df.columns))

['amount', 'city', 'founded_year', 'funding_date', 'funding_stage', 'industry', 'investors', 'startup_name']



### Q19. founded_year stats (no .describe())

In [ ]:
# Q19
years = df['founded_year'].dropna()

year_stats = {
    'count': int(years.count()),
    'mean': round(years.mean(), 2),
    'min': int(years.min()),
    'max': int(years.max()),
    'median': years.median(),
    'std': round(years.std(), 2),
}
print(year_stats)

{'count': 427, 'mean': np.float64(2016.43), 'min': 2008, 'max': 2022, 'median': np.float64(2017.0), 'std': np.float64(4.25)}



### Q20. unique cities after mapping

In [ ]:
# Q20
unique_cities = sorted(df['city'].unique())
print(unique_cities)
print("count:", len(unique_cities))

['Ahmedabad', 'Bengaluru', 'Chennai', 'Gurugram', 'Hyderabad', 'Kolkata', 'Mumbai', 'New Delhi', 'Noida', 'Pune']
count: 10



### Q21. startups per industry

In [ ]:
# Q21
print(df['industry'].value_counts())

industry
AI/ML         54
FoodTech      39
SaaS          36
BeautyTech    34
TravelTech    34
Gaming        34
PropTech      32
HealthTech    31
E-commerce    30
AgriTech      29
EV            27
Logistics     27
FinTech       22
EdTech        21
Name: count, dtype: int64



### Q22. amount column checks

In [ ]:
# Q22
print("(a) dtype:", df['amount'].dtype)
print("(b) NaN count:", df['amount'].isnull().sum())
print("(c) mean:", round(df['amount'].mean(), 2))
print("(d) min:", df['amount'].min(), " max:", df['amount'].max())

(a) dtype: float64
(b) NaN count: 24
(c) mean: 753.06
(d) min: 1.0  max: 3549.0



### Q23. deals per year

In [ ]:
# Q23
deals_per_year = df['funding_date'].dt.year.value_counts().sort_index()
print("(a) deals per year:")
print(deals_per_year)

print("\n(b) top year:", int(deals_per_year.idxmax()))

(a) deals per year:
funding_date
2008.0     2
2009.0     1
2010.0     7
2011.0     4
2012.0     7
2013.0     7
2014.0     7
2015.0     8
2016.0    21
2017.0    21
2018.0    25
2019.0    43
2020.0    39
2021.0    59
2022.0    70
2023.0    69
2024.0    42
Name: count, dtype: int64

(b) top year: 2022



### Q24. startups per funding stage

In [ ]:
# Q24
print(df['funding_stage'].value_counts())

funding_stage
Series B        56
Series E        48
Pre-Seed        48
Seed            47
Series C        46
Series A        43
Bridge          41
Pre-Series A    38
Series D        38
IPO             33
Pre-IPO         12
Name: count, dtype: int64


### Q25. Bengaluru startups > 100 Cr

In [ ]:
# Q25
bengaluru_big = df[(df['city'] == 'Bengaluru') & (df['amount'] > 100)]
bengaluru_big = bengaluru_big.sort_values('amount', ascending=False)
print(bengaluru_big[['startup_name', 'amount']])

       startup_name  amount
311        Routemap  3528.0
226     Agriconnect  3526.0
329     Whitehat Jr  3511.0
154        Ecomotor  2244.0
325       Beautybox  2239.0
98           Meesho  2231.0
146       Curefoods  2230.0
26      Mediconnect  2226.0
299     Villagemart  2195.0
236       Vitalcare  1431.0
278           Paytm  1423.0
65        Instaloan  1420.0
252  Playtime Games  1410.0
369            Cred   950.0
67      Simplilearn   923.0
225    Smartscholar   899.0
347         Flyhigh   728.0
447        Agroplus   726.0
291     Agriconnect   713.0
243        Stylehub   705.0
255      Harvestify   497.0
341          Practo   486.0
233       Vitalcare   456.0
88            Zepto   455.0
401        Shopeasy   449.0
149     Mediconnect   304.0
85        Smartshop   214.0
40         Rupaypro   186.0
182         Fetchit   150.0
434       Games24X7   118.0
398         Cuemath   115.0



### Q26. Fintech/Edtech/HealthTech count (isin)

In [ ]:
# Q26
target_industries = df[df['industry'].isin(['FinTech', 'EdTech', 'HealthTech'])]
print("count:", len(target_industries))

count: 74



### Q27. founded 2015-2020 (between)

In [ ]:
# Q27
# between() returns False for NaN automatically, so no extra dropna needed
founded_2015_2020 = df[df['founded_year'].between(2015, 2020)]
print("count:", len(founded_2015_2020))

count: 211



### Q28. names containing 'pay'

In [ ]:
# Q28
pay_startups = df[df['startup_name'].str.contains('pay', case=False, na=False)]
print(sorted(pay_startups['startup_name'].unique()))

['Paybuddy', 'Paytm', 'Quickpay India', 'Razorpay', 'Rupaypro']



### Q29. funding efficiency

In [ ]:
# Q29
df['funding_efficiency'] = df['amount'] / (2024 - df['founded_year'])

top10_efficient = df.sort_values('funding_efficiency', ascending=False).head(10)
print(top10_efficient[['startup_name', 'industry', 'amount', 'founded_year', 'funding_efficiency']])

    startup_name    industry  amount  founded_year  funding_efficiency
322   Classroomx       AI/ML  3520.0        2022.0         1760.000000
208     Cropcare  E-commerce  3518.0        2021.0         1172.666667
123        Nykaa   Logistics  3515.0        2021.0         1171.666667
154     Ecomotor      Gaming  2244.0        2022.0         1122.000000
158   Makemytrip          EV  2202.0        2022.0         1101.000000
55       Blinkit          EV  2196.0        2022.0         1098.000000
246  Fitnessplus     FinTech  3545.0        2020.0          886.250000
171     Stayeasy  E-commerce  2249.0        2021.0          749.666667
325    Beautybox  E-commerce  2239.0        2021.0          746.333333
448   Cleanslate          EV  2204.0        2021.0          734.666667



### Q30. funding tier

In [ ]:
# Q30
def get_tier(amount):
    if pd.isna(amount):
        return 'Small / Unknown'
    elif amount >= 1000:
        return 'Unicorn'
    elif amount >= 500:
        return 'Mega'
    elif amount >= 100:
        return 'Large'
    elif amount >= 10:
        return 'Mid'
    else:
        return 'Small / Unknown'

df['funding_tier'] = df['amount'].apply(lambda x: get_tier(x))
print(df['funding_tier'].value_counts())

funding_tier
Large              131
Mid                120
Unicorn             97
Mega                68
Small / Unknown     34
Name: count, dtype: int64



### Q31. founding era (pd.cut)

In [ ]:
# Q31
df['era'] = pd.cut(
    df['founded_year'],
    bins=[-np.inf, 2009, 2017, np.inf],
    labels=['Early', 'Growth', 'Recent']
)
print(df['era'].value_counts())

era
Recent    207
Growth    195
Early      25
Name: count, dtype: int64



### Q32. investor count

In [ ]:
# Q32
def count_investors(investors_str):
    if pd.isna(investors_str):
        return 0
    return len(investors_str.split(', '))

df['investor_count'] = df['investors'].apply(lambda x: count_investors(x))
print(df['investor_count'].value_counts().sort_index())

investor_count
0     10
1     94
2    199
3    118
4     29
Name: count, dtype: int64



### Q33. top 5 industries by avg funding

In [ ]:
# Q33
avg_funding_by_industry = df.groupby('industry')['amount'].mean().sort_values(ascending=False)
print(avg_funding_by_industry.head(5))

industry
FinTech       1217.636364
EdTech        1191.250000
TravelTech    1115.968750
EV            1090.769231
E-commerce    1035.107143
Name: amount, dtype: float64



### Q34. per-city metrics

In [ ]:
# Q34
city_stats = df.groupby('city').agg(
    num_startups=('startup_name', 'count'),
    total_funding=('amount', 'sum'),
    avg_funding=('amount', 'mean')
).sort_values('num_startups', ascending=False)

print(city_stats.head(10))

           num_startups  total_funding  avg_funding
city                                               
Noida                62        41499.0   715.500000
Kolkata              51        34029.0   708.937500
Bengaluru            49        39315.0   836.489362
Mumbai               45        32089.0   782.658537
Ahmedabad            44        28250.0   642.045455
Gurugram             44        33275.0   792.261905
Hyderabad            43        29738.0   743.450000
Pune                 42        33333.0   854.692308
Chennai              37        20667.0   558.567568
New Delhi            33        28608.0   953.600000



### Q35. highest avg funding, industries with 15+ startups only

In [ ]:
# Q35
industry_stats = df.groupby('industry')['amount'].agg(count='count', avg_funding='mean')
industry_stats = industry_stats[industry_stats['count'] >= 15].sort_values('avg_funding', ascending=False)

top_industry = industry_stats.iloc[0]
print(f"industry: {industry_stats.index[0]}")
print(f"count: {int(top_industry['count'])}")
print(f"avg funding: {round(top_industry['avg_funding'], 2)}")
print("\nfull table:")
print(industry_stats)

industry: FinTech
count: 22
avg funding: 1217.64

full table:
            count  avg_funding
industry                      
FinTech        22  1217.636364
EdTech         20  1191.250000
TravelTech     32  1115.968750
EV             26  1090.769231
E-commerce     28  1035.107143
Gaming         32   974.750000
FoodTech       37   813.378378
BeautyTech     32   622.531250
SaaS           35   577.485714
Logistics      25   570.960000
AI/ML          51   527.725490
PropTech       29   452.482759
HealthTech     30   412.533333
AgriTech       27   334.444444



### Q36. funding tier x top 5 industries (crosstab)

In [ ]:
# Q36
top5_industries = df['industry'].value_counts().head(5).index
subset = df[df['industry'].isin(top5_industries)]

tier_crosstab = pd.crosstab(subset['industry'], subset['funding_tier'])
print(tier_crosstab)

funding_tier  Large  Mega  Mid  Small / Unknown  Unicorn
industry                                                
AI/ML            12     7   20                6        9
BeautyTech       14     4    8                2        6
FoodTech         12     5   11                2        9
SaaS             11     4   11                3        7
TravelTech       11     7    4                2       10



### Q37. highest-funded startup per industry

In [ ]:
# Q37
top_idx = df.groupby('industry')['amount'].idxmax()
top_per_industry = df.loc[top_idx.dropna(), ['startup_name', 'industry', 'city', 'amount']]
print(top_per_industry.reset_index(drop=True))

   startup_name    industry       city  amount
0    Classroomx       AI/ML  Ahmedabad  3520.0
1     Boldfoods    AgriTech  Hyderabad  1449.0
2     Vitalcare  BeautyTech       Pune  3522.0
3       Curefit  E-commerce     Mumbai  3540.0
4         Paytm          EV       Pune  3533.0
5     Fitfusion      EdTech     Mumbai  3546.0
6   Fitnessplus     FinTech  New Delhi  3545.0
7   Simplilearn    FoodTech    Kolkata  3549.0
8     Wealthwiz      Gaming    Kolkata  3549.0
9    Investedge  HealthTech    Kolkata  2219.0
10        Nykaa   Logistics  New Delhi  3515.0
11    Mamaearth    PropTech  Hyderabad  2235.0
12  Whitehat Jr        SaaS  Hyderabad  2250.0
13     Rupaypro  TravelTech  Ahmedabad  3541.0



### Q38. emerging players

In [ ]:
# Q38
emerging_players = df[
    (df['founded_year'] >= 2019) &
    (df['amount'] >= 100) &
    (df['industry'].isin(['FinTech', 'EdTech', 'HealthTech']))
].sort_values('amount', ascending=False)

print(emerging_players[['startup_name', 'industry', 'city', 'founded_year', 'amount']])

# a VC would care about this list because these are young companies already raising big
# in hot sectors - good candidates for the next round before valuations go up more

      startup_name    industry       city  founded_year  amount
246    Fitnessplus     FinTech  New Delhi        2020.0  3545.0
57       Learnlive     FinTech      Noida        2019.0  3512.0
386     Investedge  HealthTech    Kolkata        2020.0  2219.0
178     Classroomx     FinTech      Noida        2022.0  1418.0
269        Vedantu  HealthTech      Noida        2021.0  1413.0
461        Postman      EdTech  Hyderabad        2019.0  1404.0
369           Cred      EdTech  Bengaluru        2020.0   950.0
342        Voltcar      EdTech  Hyderabad        2021.0   916.0
378  Bookmyservice  HealthTech      Noida        2021.0   902.0
417    Villagemart      EdTech   Gurugram        2021.0   896.0
370      Agriboost     FinTech  New Delhi        2021.0   729.0
449      Groomguru  HealthTech      Noida        2021.0   722.0
130      Dermaplus  HealthTech   Gurugram        2019.0   711.0
340       Fundflow     FinTech      Noida        2019.0   709.0
334       Shopeasy     FinTech   Gurugra


### Q39. investor hub (top city by total funding, 20+ startups)

In [ ]:
# Q39
city_totals = df.groupby('city').agg(count=('startup_name', 'count'), total_funding=('amount', 'sum'))
city_totals = city_totals[city_totals['count'] >= 20].sort_values('total_funding', ascending=False)

top_city = city_totals.iloc[0]
print(f"city: {city_totals.index[0]}")
print(f"count: {int(top_city['count'])}")
print(f"total funding: {top_city['total_funding']}")
print("\nfull table:")
print(city_totals)

city: Noida
count: 62
total funding: 41499.0

full table:
           count  total_funding
city                           
Noida         62        41499.0
Bengaluru     49        39315.0
Kolkata       51        34029.0
Pune          42        33333.0
Gurugram      44        33275.0
Mumbai        45        32089.0
Hyderabad     43        29738.0
New Delhi     33        28608.0
Ahmedabad     44        28250.0
Chennai       37        20667.0



### Q40. market saturation (city, industry)

In [ ]:
# Q40
saturation = df.groupby(['city', 'industry']).size().sort_values(ascending=False).head(10)
print(saturation)

# most-saturated combos = most competition. everything not on this list,
# in cities that otherwise have a decent startup scene, is where the gaps are

city       industry  
Gurugram   FoodTech      10
Bengaluru  Gaming         8
Mumbai     AI/ML          8
Pune       AI/ML          8
Ahmedabad  AI/ML          7
Kolkata    AI/ML          7
           SaaS           7
Noida      FoodTech       7
           TravelTech     7
Bengaluru  AgriTech       7
dtype: int64
